Week 13: with Neural Networks

In [ ]:

#Import Section
import numpy as np
import pandas as pd
#import matplotlib.pyplot as plt

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern, ConstantKernel
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score

from scipy.stats import norm
from scipy.stats import qmc

import torch
import torch.nn as nn
import torch.optim as optim

import warnings
warnings.filterwarnings("ignore")

In [ ]:
# Creating training tensors
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
def training_tensors(X, y, percentile=80, eps=0.05):
    X_tensor = torch.from_numpy(X).float()

    thresh = np.percentile(y, percentile)
    y_hard = (y >= thresh).astype(float)

    # Label smoothing
    y_soft = y_hard * (1 - eps) + (1 - y_hard) * eps

    y_tensor = torch.from_numpy(y_soft).float().reshape(len(X), 1)
    return X_tensor, y_tensor, y_hard


# Function for training the neural network iteratively
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
def train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser):
    
    loss_history = []
    accuracy_history = []

    for epoch in range(epochs):
        model.train()

        # Forward pass
        y_pred_nn = model(X_tensor)
        loss = criterion(y_pred_nn, y_tensor)

        # Tracking loss
        loss_history.append(loss.item())

        # Backward pass
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()

        # Tracking accuracy
        y_pred_nn_np = y_pred_nn.detach().numpy()
        y_pred_nn_class = (y_pred_nn_np >= np.percentile(y_pred_nn_np, 80)).astype(float)
        
        accuracy = accuracy_score(y_class, y_pred_nn_class)
        accuracy_history.append(accuracy)

        # Printing epochs
        if epoch % 10 == 0:
            print(f"Epoch: {epoch}, Loss: {loss}, Accuracy: {accuracy}")
    
    model.eval()

    return model, y_pred_nn, loss_history, accuracy_history

Function -1 [2-Dimensional] Load Initial Data The initial dataset consists of:

inputs: 2‑D spatial coordinates

outputs: corresponding sensor readings

In [ ]:
function1_inputs = np.load(
    '../data/week13/function_1/inputs.npy')
print("Inputs:\n", function1_inputs)

function1_outputs = np.load(
    '../data/week13/function_1/outputs.npy')
print("Outputs:\n", function1_outputs)

print("Output type:", type(function1_outputs))

# Assign to working variables
X = function1_inputs
y = function1_outputs

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.04)
epochs = 40

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(2)
X_candidates = sampler.random(2**16)

with torch.no_grad(): # Disable gradients for faster computation
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=12)

# Transforming y due to the small scale of the data
y_shift = y - np.min(y) + 1e-12
y_transformed = np.log10(y_shift)

model = gaussianP.fit(X, y_transformed)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Acquisition function 
k = 0.1

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function -2 [2-Dimensional]

In [ ]:
function2_inputs = np.load(
    '../data/week13/function_2/inputs.npy')
print("Inputs:\n", function2_inputs)

function2_outputs = np.load(
    '../data/week13/function_2/outputs.npy')
print("Outputs:\n", function2_outputs)

print("Output type:", type(function2_outputs))

# Assign to working variables
X = function2_inputs
y = function2_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.05)
epochs = 15

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(2)
X_candidates = sampler.random(2**15)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=15)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)


# Exploration temperature
k = 1.5
best_value2_scaled = np.max(y)

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function -3 [3-Dimensional] 

In [ ]:
function3_inputs = np.load(
    '../data/week13/function_3/inputs.npy')
print("Inputs:\n", function3_inputs)

function3_outputs = np.load(
    '../data/week13/function_3/outputs.npy')
print("Outputs:\n", function3_outputs)

print("Output type:", type(function3_outputs))

# Assign to working variables
X = function3_inputs
y = function3_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 60

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(3)
X_candidates = sampler.random(2**16)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# === SCALING-BASED ACQUISITION ===
n = len(X)

# Exploration temperature
k = 0.5
best_value2_scaled = np.max(y)

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function-4 [4-diemnsional] For many‑maxima, non‑smooth functions, use a Matern(ν = 1.5 or 2.5) kernel, high κ with slow decay, and inject explicit random exploration. This combination prevents early over‑commitment and keeps the search globally aware.

In [ ]:
function4_inputs = np.load(
    '../data/week13/function_4/inputs.npy')
print("Inputs:\n", function4_inputs)

function4_outputs = np.load(
    '../data/week13/function_4/outputs.npy')
print("Outputs:\n", function4_outputs)

print("Output type:", type(function4_outputs))

# Assign to working variables
X = function4_inputs
y = function4_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 35

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(4)
X_candidates = sampler.random(2**15)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * Matern(length_scale=1, nu=2.5)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# Exploration temperature
k = 1.5

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function-5 [4-diemnsional]

In [ ]:
function5_inputs = np.load(
    '../data/week13/function_5/inputs.npy')
print("Inputs:\n", function5_inputs)

function5_outputs = np.load(
    '../data/week13/function_5/outputs.npy')
print("Outputs:\n", function5_outputs)

print("Output type:", type(function5_outputs))

# Assign to working variables
X = function5_inputs
y = function5_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.03)
epochs = 20

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(4) # Using smarter sampling
X_candidates = sampler.random(2**15)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=12)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

y_func5_scaled = np.max(y_scaled)
ei = expected_improvement(mu, sigma, y_func5_scaled)
# === SCALING-BASED ACQUISITION ===
n = len(X)

# Exploration temperature
k = 0.5

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function -6 [5-Dimensional]

In [ ]:
function6_inputs = np.load(
    '../data/week13/function_6/inputs.npy')
print("Inputs:\n", function6_inputs)

function6_outputs = np.load(
    '../data/week13/function_6/outputs.npy')
print("Outputs:\n", function6_outputs)

print("Output type:", type(function6_outputs))

# Assign to working variables
X = function6_inputs
y = function6_outputs
y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 30

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(5) # Using smarter sampling
X_candidates = sampler.random(2**16)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# === SCALING-BASED ACQUISITION ===
n = len(X)

# Exploration temperature
k = 1.5

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

In [ ]:
Function -7 [6-Dimensional]

In [ ]:
function7_inputs = np.load(
    '../data/week13/function_7/inputs.npy')
print("Inputs:\n", function7_inputs)

function7_outputs = np.load(
    '../data/week13/function_7/outputs.npy')
print("Outputs:\n", function7_outputs)

print("Output type:", type(function7_outputs))

# Assign to working variables
X = function7_inputs
y = function7_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 20

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(6) # Using smarter sampling
X_candidates = sampler.random(2**16)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * Matern(length_scale=1, nu=2.5)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=9)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# === SCALING-BASED ACQUISITION ===
n = len(X)

# Exploration temperature
k = 1.0

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))

Function -8 [8-Dimensional]

In [ ]:
function8_inputs = np.load(
    '../data/week13/function_8/inputs.npy')
print("Inputs:\n", function8_inputs)

function8_outputs = np.load(
    '../data/week13/function_8/outputs.npy')
print("Outputs:\n", function8_outputs)

print("Output type:", type(function8_outputs))

# Assign to working variables
X = function8_inputs
y = function8_outputs

y_stdscaler = StandardScaler()
y_scaled = y_stdscaler.fit_transform(y.reshape(-1,1)).ravel()

In [ ]:
X_tensor, y_tensor, y_class = training_tensors(X,y)

input_dim = X.shape[1]
hidden_dim = len(X)

model = nn.Sequential(
    nn.Linear (input_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear (hidden_dim, hidden_dim),
    nn.ReLU(),
    nn.Linear(hidden_dim,1),
    nn.Sigmoid()
)

# Define Hyperparameters
criterion = nn.BCELoss()
optimiser = optim.Adam(model.parameters(), lr=0.02)
epochs = 20

# Running model through iterative cycle
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
model, y_pred_nn, loss_history, accuracy_history = train_nn(X_tensor, y_tensor, y_class, model, epochs, criterion, optimiser)


# Creating X candidates and predicting
# ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
sampler = qmc.Sobol(8) # Using smarter sampling
X_candidates = sampler.random(2**15)

with torch.no_grad(): 
    X_candidates_tensor = torch.from_numpy(X_candidates).float()

    X_candidates_prob_tensor = model(X_candidates_tensor)
    X_candidates_prob = X_candidates_prob_tensor.cpu().numpy().ravel()

In [ ]:
# Define GaussianProcess model and fitting to data
kernel = ConstantKernel(1.0) * RBF(length_scale=1)
gaussianP = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=12)

model = gaussianP.fit(X, y_scaled)

#sampler = qmc.Sobol(4) # Using smarter sampling for 4-diemnsions
#X_candidates = sampler.random(2**15)

# GaussianProcess predictions
mu, sigma = model.predict(X_candidates, return_std=True)

# === SCALING-BASED ACQUISITION ===
n = len(X)

# Exploration temperature
k = 1.0

improvement = (mu - np.max(y))
Z = improvement / (sigma + 1e-9)
ei = improvement * norm.cdf(Z) + sigma * norm.pdf(Z)
ei[sigma == 0] = 0

acquisition = ei * (0.5 + 0.5 * X_candidates_prob)
idx_next = np.argmax(acquisition)

x_next = X_candidates[idx_next]

print('The next query point is:', np.round(x_next, 6))